In a FastAPI project, choosing between an **RDBMS (SQL)** and a **NoSQL Database** comes down to **data structure predictability, query complexity, and transaction safety**. FastAPI itself is database-agnostic—its asynchronous nature (`async/await`) works exceptionally well with both SQL (via `SQLAlchemy 2.0` / `asyncpg`) and NoSQL (via `Beanie` / `Motor` for MongoDB).

Here is the decision matrix for choosing the right database for your FastAPI backend.

---

### **1. When to Choose a SQL Database (PostgreSQL, MySQL, SQLite)**

SQL is the default choice when your data has clear relationships and requires strict data consistency.

#### **Choose SQL if:**

* **Your Data is Highly Relational:** You have entities that link together (e.g., `Users` $\rightarrow$ `Orders` $\rightarrow$ `Products` $\rightarrow$ `Payments`).
* **You Require ACID Compliance:** Financial transactions, inventory updates, or sensitive operations where data corruption or partial writes are unacceptable.
* **Your Data Schema is Stable:** Your data structures don't change drastically every week, and you want the database to enforce structure.
* **Complex Aggregations & Joins:** You frequently run queries with multi-table `JOIN`s, filtering on deep relationships, or window functions.

#### **FastAPI Stack for SQL:**

* **ORM / Query Builder:** `SQLAlchemy 2.0` (with `asyncio` support) or `SQLModel` (built by the author of FastAPI to unify Pydantic and SQLAlchemy models).
* **Driver:** `asyncpg` (for PostgreSQL) or `aiomysql` (for MySQL).
* **Migrations:** `Alembic` (essential for schema changes).

```python
# SQLModel / Pydantic integration in FastAPI
from sqlmodel import SQLModel, Field, Relationship
from typing import Optional, List

class User(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    email: str = Field(unique=True, index=True)
    orders: List["Order"] = Relationship(back_populates="user")
```

---

### **2. When to Choose a NoSQL Database (MongoDB, Redis, Cassandra, DynamoDB)**

NoSQL shines when dealing with unstructured data, high write speeds, dynamic attributes, or key-value lookups.

#### **Choose NoSQL if:**

* **Flexible or Evolving Schemas:** You are handling nested documents, JSON logs, user settings, or e-commerce products where different items have completely different attributes (e.g., shoes have sizes, electronics have voltage).
* **High-Volume Simple Writes/Reads:** Real-time analytics, event tracking, IoT data streams, or chat messages.
* **Key-Value Caching & Session Storage:** Storing transient data like JWT blocklists, rate-limiting counters, or session states (typically using **Redis**).
* **Native JSON Alignment:** Pydantic models naturally dump directly into BSON/JSON structures without complex ORM translation.

#### **FastAPI Stack for NoSQL (MongoDB example):**

* **ODM (Object Document Mapper):** `Beanie` (Pydantic-based async ODM) or `MongoEngine`.
* **Async Driver:** `Motor` (Official async MongoDB driver).

```python
# Beanie (MongoDB + Pydantic) in FastAPI
from beanie import Document
from pydantic import Field

class Product(Document):
    name: str
    price: float
    attributes: dict  # Dynamic attributes fit natively here!
    
    class Settings:
        name = "products"
```

---

### **🧠 The Decision Flowchart**

```text
                           Is your data strictly structured 
                             & heavily interconnected?
                                      │
                     ┌────────────────┴────────────────┐
                    YES                                NO
                     │                                 │
                     ▼                                 ▼
           Do you need strict ACID            Is it dynamic, unstructured,
           transactions & JOINS?              or flat document data?
                     │                                 │
            ┌────────┴────────┐               ┌────────┴────────┐
           YES                NO             YES                NO
            │                  │              │                  │
            ▼                  ▼              ▼                  ▼
     Use PostgreSQL       Use SQLite     Use MongoDB /      Use Redis
      (SQL Database)     (Local/Dev)      DynamoDB        (Key-Value Cache)
```

---

### **⚖️ Feature-by-Feature Comparison**

| Criteria | SQL (e.g., PostgreSQL) | NoSQL (e.g., MongoDB / Redis) |
| --- | --- | --- |
| **FastAPI Integration** | Excellent (`SQLModel`, `SQLAlchemy 2.0`) | Flawless (Matches Pydantic JSON structure) |
| **Async Performance** | Blazing fast with `asyncpg` driver | Excellent with `Motor` / `redis-py` |
| **Schema Flexibility** | Rigid (Requires DB migrations via `Alembic`) | Flexible (Add fields dynamically without migrations) |
| **Query Complexity** | High (`JOIN`s, subqueries, group operations) | Moderate to Low (Optimized for document lookups) |
| **Data Integrity** | Enforced at the Database level (Foreign Keys) | Enforced at the Application level (Pydantic models) |

---

### **💡 The Hybrid Approach (Production Best Practice)**

- In modern web development, you rarely have to choose only one. The most common enterprise architecture pairs them together:
    * **PostgreSQL:** Serves as the **Primary Source of Truth** for core entities (Users, Accounts, Relational Data).
    * **Redis (NoSQL):** Sits in front of PostgreSQL as an **In-Memory Cache & Session Store** to intercept frequent reads and enforce rate limits.
    * **MongoDB (NoSQL):** Stores **Logs, User Activity Streams, or Unstructured Product Catalogs**.

# **SQLAlchemy**

SQLAlchemy is the premier Python SQL toolkit. It acts as a bridge between your Python code and your relational database, handling connection management, query construction, and data mapping.

Instead of writing raw SQL strings (which are prone to SQL injection and typos), SQLAlchemy lets you interact with your database using Python objects and expressions.

---

## **1. SQLAlchemy Core vs. SQLAlchemy ORM**

SQLAlchemy is designed as a two-layer system: **Core** (the foundational abstraction layer) and **ORM** (the high-level object mapper built on top of Core).

```text
┌────────────────────────────────────────────────────────┐
│                    SQLAlchemy ORM                      │
│   (Classes, Objects, Unit of Work, Relational Mapping) │
└───────────────────────────┬────────────────────────────┘
                            │ Built on top of
                            ▼
┌────────────────────────────────────────────────────────┐
│                    SQLAlchemy Core                     │
│    (Schema Definition, SQL Expression Language, Engine)│
└───────────────────────────┬────────────────────────────┘
                            │ Communicates with
                            ▼
┌────────────────────────────────────────────────────────┐
│                 DBAPI / Async Driver                   │
│               (e.g., asyncpg, psycopg)                 │
└────────────────────────────────────────────────────────┘

```

### **SQLAlchemy Core**

* **What it is:** A database abstraction layer centered around SQL tables and raw expressions. It mimics standard SQL structure directly in Python.
* **How it works:** You query against `Table` objects and receive `Row` tuple-like objects back.
* **Best for:** Complex data pipelines, raw reporting queries, or microservices where you don't need rich domain model classes.

### **SQLAlchemy ORM (Object-Relational Mapper)**

* **What it is:** A high-level pattern built on top of Core that maps database tables to custom Python classes (Domain Models).
* **How it works:** You interact with Python class instances. It includes a **Unit of Work** pattern (`Session`) that automatically tracks changes to objects and flushes them to the database in single transactions.
* **Best for:** Rich application backends (like FastAPI APIs) with business logic, cross-table relationships, and CRUD endpoints.

### **Comparison Table**

| Feature | SQLAlchemy Core | SQLAlchemy ORM |
| --- | --- | --- |
| **Primary Abstraction** | Tables, Columns, SQL Expressions | Python Classes, Class Instances |
| **Query Style** | `select(users_table).where(...)` | `select(User).where(...)` |
| **Return Types** | Database Rows (`Row`) | Model Objects (`User`) |
| **State Tracking** | None (manual SQL statements) | Automatic via `Session` / `AsyncSession` |
| **Relationship Management** | Manual foreign key joins | Declarative `relationship()` hooks |

---

## **2. Integrating Async SQLAlchemy 2.0 with FastAPI**

FastAPI relies heavily on `async/await`. Using **SQLAlchemy 2.0** with an asynchronous driver (like `asyncpg` for PostgreSQL or `aiosqlite` for SQLite) ensures database operations do not block the event loop.

### **Step 1: Database Setup (`database.py`)**

Set up the async engine and session maker:

```python
from sqlalchemy.ext.asyncio import create_async_engine, async_sessionmaker, AsyncSession
from sqlalchemy.orm import DeclarativeBase

# 1. Define the Async Database URL (using SQLite async driver here)
DATABASE_URL = "sqlite+aiosqlite:///./test.db"

# 2. Create the Async Engine
engine = create_async_engine(DATABASE_URL, echo=True)

# 3. Create a Session Factory
AsyncSessionLocal = async_sessionmaker(
    bind=engine,
    class_=AsyncSession,
    expire_on_commit=False
)

# 4. Base class for ORM Models
class Base(DeclarativeBase):
    pass

# 5. FastAPI Dependency to manage session lifecycle per request
async def get_db():
    async with AsyncSessionLocal() as session:
        yield session
```

---

### **Step 2: Define ORM Model (`models.py`)**

Create your ORM table mapping:

```python
from sqlalchemy import String, Integer
from sqlalchemy.orm import Mapped, mapped_column
from database import Base

class ItemModel(Base):
    __tablename__ = "items"

    id: Mapped[int] = mapped_column(Integer, primary_key=True, index=True)
    title: Mapped[str] = mapped_column(String, index=True)
    description: Mapped[str] = mapped_column(String, default="")
```

---

### **Step 3: Define Pydantic Schemas (`schemas.py`)**

Keep input validation separated from database models:

```python
from pydantic import BaseModel

class ItemCreate(BaseModel):
    title: str
    description: str = ""

class ItemResponse(ItemCreate):
    id: int

    class Config:
        from_attributes = True  # Allows Pydantic to parse ORM objects
```

---

### **Step 4: Build FastAPI Routes (`main.py`)**

Inject `AsyncSession` using `Depends(get_db)` into your path operations:

```python
from fastapi import FastAPI, Depends, HTTPException, status
from sqlalchemy.ext.asyncio import AsyncSession
from sqlalchemy import select
from typing import List

from database import engine, Base, get_db
import models
import schemas

app = FastAPI()

# Create tables on startup (In production, use Alembic migrations instead)
@app.on_event("startup")
async def startup():
    async with engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)

# CREATE Item
@app.post("/items", response_model=schemas.ItemResponse, status_code=status.HTTP_201_CREATED)
async def create_item(item: schemas.ItemCreate, db: AsyncSession = Depends(get_db)):
    db_item = models.ItemModel(title=item.title, description=item.description)
    db.add(db_item)
    await db.commit()
    await db.refresh(db_item)
    return db_item

# READ All Items
@app.get("/items", response_model=List[schemas.ItemResponse])
async def read_items(db: AsyncSession = Depends(get_db)):
    result = await db.execute(select(models.ItemModel))
    items = result.scalars().all()
    return items

# READ Single Item by ID
@app.get("/items/{item_id}", response_model=schemas.ItemResponse)
async def read_item(item_id: int, db: AsyncSession = Depends(get_db)):
    result = await db.execute(select(models.ItemModel).where(models.ItemModel.id == item_id))
    item = result.scalar_one_or_none()
    
    if item is None:
        raise HTTPException(status_code=404, detail="Item not found")
    return item
```